# **Import Libraries and dataset**

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
import seaborn as sns

import warnings
warnings.filterwarnings('ignore')

In [2]:
df = pd.read_csv("churndataset.csv")

df.head()


,CustomerID,Churn,Tenure,PreferredLoginDevice,CityTier,WarehouseToHome,PreferredPaymentMode,Gender,HourSpendOnApp,NumberOfDeviceRegistered,SatisfactionScore,MaritalStatus,NumberOfAddress,Complain,OrderAmountHikeFromlastYear,CouponUsed,OrderCount,DaySinceLastOrder,CashbackAmount,PreferedOrderCat
0,50001,1,4.0,Mobile Phone,3,6.0,Debit Card,Female,3.0,3,2,Single,9,1,11.0,1.0,1.0,5.0,160,Power Backup
1,50002,1,NaN,Phone,1,8.0,UPI,Male,3.0,4,3,Single,7,1,15.0,0.0,1.0,0.0,121,Washing Machines & Dryers
2,50003,1,NaN,Phone,1,30.0,Debit Card,Male,2.0,4,3,Single,6,1,14.0,0.0,1.0,3.0,120,Power Backup
3,50004,1,0.0,Phone,3,15.0,Debit Card,Male,2.0,4,5,Single,8,0,23.0,0.0,1.0,3.0,134,Heating Appliances
4,50005,1,0.0,Phone,1,12.0,CC,Male,NaN,3,5,Single,3,0,11.0,1.0,1.0,3.0,130,Washing Machines & Dryers


# **Data Cleaning**

## **Basic Info**

In [3]:
# Basic information on the dataset
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5630 entries, 0 to 5629
Data columns (total 20 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   CustomerID                   5630 non-null   int64  
 1   Churn                        5630 non-null   int64  
 2   Tenure                       5366 non-null   float64
 3   PreferredLoginDevice         5630 non-null   object 
 4   CityTier                     5630 non-null   int64  
 5   WarehouseToHome              5379 non-null   float64
 6   PreferredPaymentMode         5630 non-null   object 
 7   Gender                       5630 non-null   object 
 8   HourSpendOnApp               5375 non-null   float64
 9   NumberOfDeviceRegistered     5630 non-null   int64  
 10  SatisfactionScore            5630 non-null   int64  
 11  MaritalStatus                5630 non-null   object 
 12  NumberOfAddress              5630 non-null   int64  
 13  Complain          

**OBSERVATIONS**
- 5630 entries and 20 columns
- **Churn** is our target variable
- 8 floating point variables, 7 integer variables and 5 string variables
- There are some missing values.

**Check for duplicate rows** -

In [4]:
df.duplicated().sum()

np.int64(0)

No duplicates

## **Check for Inconsistancies and corruptness**

In [5]:
# Find the unique number of values for each variable
df.nunique()

CustomerID                     5630
Churn                             2
Tenure                           36
PreferredLoginDevice              3
CityTier                          3
WarehouseToHome                  34
PreferredPaymentMode              7
Gender                            2
HourSpendOnApp                    6
NumberOfDeviceRegistered          6
SatisfactionScore                 5
MaritalStatus                     3
NumberOfAddress                  15
Complain                          2
OrderAmountHikeFromlastYear      16
CouponUsed                       17
OrderCount                       16
DaySinceLastOrder                22
CashbackAmount                  220
PreferedOrderCat                  6
dtype: int64

Except for Customer ID and CashbackAmount, lets find the unique values for the other columns

In [6]:
# Unique values for columns
for col in df.drop(['CustomerID', 'CashbackAmount'], axis=1).columns:
  print('Column Name :', col)
  print(df[col].unique())
  print('-------------------------------------------------------------')

Column Name : Churn
[1 0]
-------------------------------------------------------------
Column Name : Tenure
[ 4. nan  0. 13. 11.  9. 19. 20. 14.  8. 18.  5.  2. 30.  1. 23.  3. 29.
  6. 26. 28.  7. 24. 25. 10. 15. 22. 27. 16. 12. 21. 17. 50. 60. 31. 51.
 61.]
-------------------------------------------------------------
Column Name : PreferredLoginDevice
['Mobile Phone' 'Phone' 'Computer']
-------------------------------------------------------------
Column Name : CityTier
[3 1 2]
-------------------------------------------------------------
Column Name : WarehouseToHome
[  6.   8.  30.  15.  12.  22.  11.   9.  31.  18.  13.  20.  29.  28.
  26.  14.  nan  10.  27.  17.  23.  33.  19.  35.  24.  16.  25.  32.
  34.   5.  21. 126.   7.  36. 127.]
-------------------------------------------------------------
Column Name : PreferredPaymentMode
['Debit Card' 'UPI' 'CC' 'Cash on Delivery' 'E wallet' 'COD' 'Credit Card']
-------------------------------------------------------------
Column 

**OBSERVATIONS :** We can observe the following inconsistancies:
- **PreferredLoginDevice** : 'Mobile Phone' and 'Mobile' probably indicates mobile phone, since you cannot login with a land phone -> change it to a common 'Mobile'
- **WarehouseToHome** : Has extremely large values like 126 and 127, which is highly unlikely since people don't usually travel that long distance to buy goods. The values might be mistakenly entered -> change 126->26 and 127->27
- **PreferredPaymentMode** : 
  - 'CC' and 'Credit Card' indicate credit card -> change it to 'CC'
  - 'Cash on Delivery' and 'COD' indicate cash on delivery -> change it to 'COD'
  - In addition, change Debit Card to DC for convenience.
- **PreferedOrderCat**:
  - 'Mobile' and 'Mobile Phone' -> change it to 'Mobile'
  - 'Laptop & Accessory' -> change it to 'Laptop' for convenience.
- **NumberOfAddress** : Has large numbers above 10, which seems very unlikely. Need to look into it.

In [7]:
# Correct values of column PreferredLoginDevice, WarehouseToHome, PreferredPaymentMode and PreferedOrderCat
df['PreferredLoginDevice'].replace({'Mobile Phone':'Mobile', 'Phone':'Mobile'}, inplace=True)
df['WarehouseToHome'].replace({126:26, 127:27}, inplace=True)
df['PreferredPaymentMode'].replace({'Credit Card':'CC', 'Cash on Delivery':'COD', 'Debit Card':'DC' }, inplace=True)
df['PreferedOrderCat'].replace({'Mobile Phone':'Mobile', 'Laptop & Accessory':'Laptop'}, inplace=True)

Now lets delve into the problem with the inconsistant values of NumberOfAddress.

In [8]:
# Number of rows with NumberOfAddress greater than 5
df[df.NumberOfAddress>5].shape[0]

1453

There seems to be 1453 customers with more than 5 addresses registered in the ecommerce platform, in their name. This seems very unlikely. But we cannot rule out the following possibilities:
- The customer might have multiple properties(houses) in their name.
- Slight change in the address text.
- Addresses of workplace/workplaces if the customer works shifts.
- Ordering for someone else
- Someone else must be ordering in the customer's name name(either other family member with the customer's knowledge or in the rare case fraud transactions by outsiders).

In the above mention cases, multiple addresses might be added. 

If we have some other information like the actual list of addresses registered or the locations where the orders were made, number of family members, number of children, occupation related information, annual income etc., we might get a little bit insight into the possibility of customers have multiple addresses.

However, since we do not have them, we need to explore with the available information and reach a conclusion. We shall analyse more in the EDA process and come to a conclusion.

## **Dealing with missing values**

In [9]:
# Find columns with missing values
missing_cols = [col for col in df.columns if df[col].isnull().any()]
missing_cols

['Tenure',
 'WarehouseToHome',
 'HourSpendOnApp',
 'OrderAmountHikeFromlastYear',
 'CouponUsed',
 'OrderCount',
 'DaySinceLastOrder']

In [10]:
# Find the % of missing value in the above list of columns
percent = df[missing_cols].isnull().sum() * 100/len(df)
missing_df = pd.DataFrame({'column':missing_cols,
                           'no. of missing values' : df[missing_cols].isnull().sum().values,
                           'percentage' : percent.values})
missing_df

,column,no. of missing values,percentage
0,Tenure,264,4.689165
1,WarehouseToHome,251,4.458259
2,HourSpendOnApp,255,4.529307
3,OrderAmountHikeFromlastYear,265,4.706927
4,CouponUsed,256,4.547069
5,OrderCount,258,4.582593
6,DaySinceLastOrder,307,5.452931


We see that the percent of missing values for any column is below 6%. We can either drop them (columnwise/rowwise) or impute them using any suitable method or a combination of both.

Dropping the column is not recommended, since each column has a max of only 6% missing values.

Lets look at the number of missing values in each row. If any rows has 4 or more than 4 missing values, we shall drop those rows.

In [11]:
# Calculate the number of missing values in each row and finding the count
df.isnull().sum(axis=1).value_counts()

0    3774
1    1856
Name: count, dtype: int64

The maximum number of missing value in any given row is 1. So, its difficult to drop rows with losing substantial information from the data. Therefore we shall impute them.

In our case, all our missing data are numeric type. Therefore, we shall fill the missing values with mean or median of the variable, depending in the skewness value.

In [12]:
# Calculate skewness of missing value columns
df[missing_cols].skew(axis=0, skipna=True, numeric_only=True)

Tenure                         0.736513
WarehouseToHome                0.898406
HourSpendOnApp                -0.027213
OrderAmountHikeFromlastYear    0.790785
CouponUsed                     2.545653
OrderCount                     2.196414
DaySinceLastOrder              1.191000
dtype: float64

We can see that 
- **HourSpendOnApp** is normally distributed -> Fill missing values with mean
- **Tenure and OrderAmountHikeFromlastYear** are moderately skewed -> Fill missing values with mean
- **WarehouseToHome, CouponUsed, OrderCount and DaySinceLastOrder** are highly skewed -> Fill missing values with median

In [13]:
round(df.HourSpendOnApp.mean()), round(df.Tenure.mean()), round(df.OrderAmountHikeFromlastYear.mean()), round(df.WarehouseToHome.mean())

(3, 10, 16, 16)

In [14]:
df.CouponUsed.median(), df.OrderCount.median(), df.DaySinceLastOrder.median()

(np.float64(1.0), np.float64(2.0), np.float64(3.0))

In [15]:
# Fill missing values ith mean and median
df.fillna({'HourSpendOnApp':round(df.HourSpendOnApp.mean()), 
           'Tenure':round(df.Tenure.mean()), 
           'OrderAmountHikeFromlastYear':round(df.OrderAmountHikeFromlastYear.mean()),
           'WarehouseToHome':round(df.WarehouseToHome.mean()), 
           'CouponUsed':df.CouponUsed.median(), 
           'OrderCount':df.OrderCount.median(), 
           'DaySinceLastOrder':df.DaySinceLastOrder.median()}, inplace=True)

In [16]:
df.head()

,CustomerID,Churn,Tenure,PreferredLoginDevice,CityTier,WarehouseToHome,PreferredPaymentMode,Gender,HourSpendOnApp,NumberOfDeviceRegistered,SatisfactionScore,MaritalStatus,NumberOfAddress,Complain,OrderAmountHikeFromlastYear,CouponUsed,OrderCount,DaySinceLastOrder,CashbackAmount,PreferedOrderCat
0,50001,1,4.0,Mobile,3,6.0,DC,Female,3.0,3,2,Single,9,1,11.0,1.0,1.0,5.0,160,Power Backup
1,50002,1,10.0,Mobile,1,8.0,UPI,Male,3.0,4,3,Single,7,1,15.0,0.0,1.0,0.0,121,Washing Machines & Dryers
2,50003,1,10.0,Mobile,1,30.0,DC,Male,2.0,4,3,Single,6,1,14.0,0.0,1.0,3.0,120,Power Backup
3,50004,1,0.0,Mobile,3,15.0,DC,Male,2.0,4,5,Single,8,0,23.0,0.0,1.0,3.0,134,Heating Appliances
4,50005,1,0.0,Mobile,1,12.0,CC,Male,3.0,3,5,Single,3,0,11.0,1.0,1.0,3.0,130,Washing Machines & Dryers


In [17]:
df.isnull().sum()

CustomerID                     0
Churn                          0
Tenure                         0
PreferredLoginDevice           0
CityTier                       0
WarehouseToHome                0
PreferredPaymentMode           0
Gender                         0
HourSpendOnApp                 0
NumberOfDeviceRegistered       0
SatisfactionScore              0
MaritalStatus                  0
NumberOfAddress                0
Complain                       0
OrderAmountHikeFromlastYear    0
CouponUsed                     0
OrderCount                     0
DaySinceLastOrder              0
CashbackAmount                 0
PreferedOrderCat               0
dtype: int64

# **Save the data**

In [18]:
df.to_csv("cleaned_data.csv", index=False)
